In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import prepare_data
import statsmodels.api as sm
from IPython.display import display
from plotly.subplots import make_subplots
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


def rolling_collinearity(X: pd.DataFrame, window: int = 252, step: int = 1):
    """
    係数ローリング回帰と同一窓で VIF・条件数を時系列算出。
    X: 説明変数のみ（被説明変数は含めない）
    """
    cols = X.columns.tolist()
    idx, vif_rows, cond_rows = [], [], []

    for end in range(window, len(X) + 1, step):
        win = X.iloc[end - window : end]
        # 窓内に分散ゼロ列がないか確認（祝日ゼロ埋め等の残存対策）
        if (win.std() == 0).any():
            continue
        Xc = sm.add_constant(win)
        vifs = [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
        vif_rows.append(dict(zip(Xc.columns, vifs, strict=False)))

        Xs = (win - win.mean()) / win.std()
        cond_rows.append(np.linalg.cond(sm.add_constant(Xs).values))
        idx.append(X.index[end - 1])

    vif_ts = pd.DataFrame(vif_rows, index=idx).drop(columns="const")
    cond_ts = pd.Series(cond_rows, index=idx, name="cond_number")
    return vif_ts, cond_ts


def rolling_correlation(X: pd.DataFrame, window: int = 256, step: int = 1):
    """
    説明変数間の全ペア相関を、係数ローリング回帰と同一窓で時系列算出。
    X: 説明変数のみ（被説明変数は含めない）
    戻り値: corr_ts (各列が変数ペア, 値はその窓のpairwise相関)
    """
    from itertools import combinations

    cols = X.columns.tolist()
    pairs = list(combinations(cols, 2))
    idx, rows = [], []

    for end in range(window, len(X) + 1, step):
        win = X.iloc[end - window : end]
        if (win.std() == 0).any():  # 分散ゼロ列（祝日ゼロ埋め残存等）を回避
            continue
        c = win.corr()
        rows.append({f"{a} × {b}": c.loc[a, b] for a, b in pairs})
        idx.append(X.index[end - 1])

    return pd.DataFrame(rows, index=idx)


def run_rolling_factor_regression(
    df_daily_return,
    window=252,
    maxlags=21,
    y_col="Quality",
    x_cols=("Value", "Size", "Momentum", "Low Volatility", "Growth"),
):
    X = sm.add_constant(df_daily_return[list(x_cols)])
    y = df_daily_return[y_col]

    rolling_results = []
    dates = df_daily_return.index

    for i in range(window, len(df_daily_return) + 1):
        X_slice = X.iloc[i - window : i]
        y_slice = y.iloc[i - window : i]
        current_date = dates[i - 1]

        try:
            model = sm.OLS(y_slice, X_slice).fit(
                cov_type="HAC", cov_kwds={"maxlags": maxlags}
            )
            row_data = {"Date": current_date, "R_squared": model.rsquared}
            # 各変数のcoefとz-value(t-value)を格納
            for col in X.columns:
                row_data[f"coef_{col}"] = model.params[col]
                row_data[f"zstat_{col}"] = model.bse[col]

            rolling_results.append(row_data)
        except Exception as e:
            print(f"Error at {current_date}: {e}")
            continue

    df_rolling = pd.DataFrame(rolling_results).set_index("Date")
    return df_rolling


PRJ_DIR = Path().cwd()
parquet_file = PRJ_DIR / "MSCI ACWI_FTW-LS-cum.parquet"
df_cum = prepare_data.get_factor_cum_return(parquet_file)
display(df_cum.head())

df_daily_return = prepare_data.arithmetic_cumret_to_daily(parquet_file)
display(df_daily_return.head())

variable,FTW MXWD Index 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Communications Growth Long-Short (High-Low) Total Return,FTW MXWD Index Communications Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Communications Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Communications Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Communications Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Communications Quality Long-Short (High-Low) Total Return,FTW MXWD Index Communications Size Long-Short (High-Low) Total Return,...,FTW MXWD Index Utilities Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Growth Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Utilities Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Quality Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Size Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Value Long-Short (Low-High) Total Return,FTW MXWD Index Value Long-Short (Low-High) Total Return
Date,,,,,,,,,,,,,,,,,,,,,
2007-01-01,0.00,NaN,NaN,NaN,NaN,0.00,0.00,NaN,NaN,NaN,...,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0.00,NaN,-0.00
2007-01-02,-0.09,0.00,0.00,0.00,-0.00,0.13,-0.07,NaN,0.00,0.00,...,0.00,0.00,-0.00,0.54,0.00,NaN,0.78,0.03,-0.00,0.12
2007-01-03,-0.47,0.03,-0.25,0.06,-0.29,0.62,-0.14,NaN,-0.59,0.61,...,0.85,0.24,0.33,0.13,-0.01,NaN,-0.28,-0.57,-1.59,0.03
2007-01-04,-1.01,-0.25,-0.67,1.62,-0.54,1.37,-0.41,NaN,-0.89,1.18,...,1.14,-0.86,-1.35,-0.49,-0.40,NaN,-0.52,-0.16,-3.21,-0.18
2007-01-05,-0.91,-0.03,-1.11,2.59,-1.12,1.57,-1.05,NaN,-1.10,1.34,...,1.34,-0.56,-1.65,-2.63,-2.87,NaN,-0.89,-0.89,-4.41,-0.33


[情報] 全ゼロ行(非取引日)を 1670 件除去する。
[完了] 5420 行。 中央絶対値 0.00330(小数)。 期間 2007-01-02 〜 2026-05-29。


variable,FTW MXWD Index 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Communications Growth Long-Short (High-Low) Total Return,FTW MXWD Index Communications Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Communications Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Communications Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Communications Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Communications Quality Long-Short (High-Low) Total Return,FTW MXWD Index Communications Size Long-Short (High-Low) Total Return,...,FTW MXWD Index Utilities Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Growth Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Utilities Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Quality Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Size Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Value Long-Short (Low-High) Total Return,FTW MXWD Index Value Long-Short (Low-High) Total Return
Date,,,,,,,,,,,,,,,,,,,,,
2007-01-02,-0.0009,NaN,NaN,NaN,NaN,0.0013,-0.0007,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0054,NaN,NaN,0.0078,0.0003,NaN,0.0012
2007-01-03,-0.0038,0.0003,-0.0025,0.0006,-0.0029,0.0049,-0.0007,NaN,-0.0059,0.0061,...,0.0085,0.0024,0.0033,-0.0041,-0.0001,NaN,-0.0106,-0.0060,-0.0159,-0.0009
2007-01-04,-0.0054,-0.0028,-0.0042,0.0156,-0.0025,0.0075,-0.0027,NaN,-0.0030,0.0057,...,0.0029,-0.0110,-0.0168,-0.0062,-0.0039,NaN,-0.0024,0.0041,-0.0162,-0.0021
2007-01-05,0.0010,0.0022,-0.0044,0.0097,-0.0058,0.0020,-0.0064,NaN,-0.0021,0.0016,...,0.0020,0.0030,-0.0030,-0.0214,-0.0247,NaN,-0.0037,-0.0073,-0.0120,-0.0015
2007-01-06,0.0000,0.0000,0.0000,0.0000,-0.0000,0.0000,0.0000,NaN,0.0000,0.0000,...,0.0000,0.0000,-0.0000,0.0000,0.0000,NaN,0.0000,0.0000,-0.0000,-0.0000


## クオリティLS Cumlative Return


In [2]:
quality_cols = [c for c in df_cum.columns if "Quality" in c]
df_cum_quality = df_cum[quality_cols].copy()
df_cum_quality.rename(
    columns={
        k: v
        for k, v in zip(
            quality_cols,
            [
                c.replace("FTW MXWD Index ", "").replace(
                    " Quality Long-Short (High-Low) Total Return", ""
                )
                for c in quality_cols
            ],
            strict=False,
        )
    },
    inplace=True,
)

fig = go.Figure()
for col in df_cum_quality.columns:
    fig.add_trace(
        go.Scatter(
            x=df_cum_quality.index, y=df_cum_quality[col], mode="lines", name=col
        )
    )
fig.update_layout(
    title="Quality LS Total Return",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=40, t=50, b=40, r=40),
)
fig.update_xaxes(title="Date")
fig.show()

## ベース多変量回帰

$$ R*{Quality,t} = \alpha + \sum*{factor \in Val, Grwoth, Size, Mom, LowVol} \beta*{factor} R*{factor,t} + \epsilon_t $$


In [4]:
def get_quality_return_df(
    df_daily_return: pd.DataFrame,
    target_sector: str | None = None,
    factors: list[str] | None = None,
) -> pd.DataFrame:
    if not factors:
        factors = [
            "Momentum",
            "Low Volatility",
            "Size",
            "Value",
            "Growth Long-Short",
            "Quality",
        ]

    if not target_sector:
        target_cols = [
            "FTW MXWD Index " + f + " (High-Low) Total Return"
            if f == "Growth Long-Short"
            else "FTW MXWD Index " + f + " Long-Short (High-Low) Total Return"
            for f in factors
        ]
        target_cols = [
            f.replace("High-Low", "Low-High")
            if any(factor in f for factor in ["Low Volatility", "Value"])
            else f
            for f in target_cols
        ]
    else:
        key_cols = f"FTW MXWD Index {target_sector}"
        target_cols = [
            s
            for s in df_daily_return.columns
            if (key_cols in s) and any(f in s for f in factors)
        ]

    ret_quality = df_daily_return[target_cols].copy()

    ret_quality.columns = [
        s.replace(f"FTW MXWD Index {target_sector} ", "")
        .replace("FTW MXWD Index ", "")
        .replace(" Long-Short (High-Low) Total Return", "")
        .replace(" Long-Short (Low-High) Total Return", "")
        .replace("Sector Neutralized ", "")
        for s in ret_quality.columns
    ]
    ret_quality = ret_quality[~(ret_quality == 0).all(axis=1).copy()].dropna(how="any")

    return ret_quality

In [8]:
target_sector = "Sector Neutralized"
display(
    df_daily_return[
        [
            "FTW MXWD Index Sector Neutralized Growth Long-Short (High-Low) Total Return",
            "FTW MXWD Index Sector Neutralized Value Long-Short (Low-High) Total Return",
            "FTW MXWD Index Sector Neutralized Low Volatility Long-Short (Low-High) Total Return",
            "FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return",
            "FTW MXWD Index Sector Neutralized Momentum Long-Short (High-Low) Total Return",
            "FTW MXWD Index Sector Neutralized Size Long-Short (High-Low) Total Return",
        ]
    ]
)
ret_quality = get_quality_return_df(
    df_daily_return=df_daily_return, target_sector=target_sector
)
display(ret_quality)

variable,FTW MXWD Index Sector Neutralized Growth Long-Short (High-Low) Total Return,FTW MXWD Index Sector Neutralized Value Long-Short (Low-High) Total Return,FTW MXWD Index Sector Neutralized Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return,FTW MXWD Index Sector Neutralized Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Sector Neutralized Size Long-Short (High-Low) Total Return
Date,,,,,,
2007-01-02,-0.0002,0.0026,0.0009,-0.0001,0.0005,-0.0016
2007-01-03,-0.0011,-0.0042,-0.0063,-0.0029,-0.0039,-0.0039
2007-01-04,-0.0011,-0.0047,-0.0069,-0.0017,-0.0073,-0.0002
2007-01-05,0.0019,-0.0021,-0.0027,0.0005,-0.0043,-0.0028
2007-01-06,0.0000,-0.0000,-0.0000,0.0000,0.0000,0.0000
...,...,...,...,...,...,...
2026-05-25,0.0057,0.0055,0.0126,-0.0012,0.0079,0.0000
2026-05-26,0.0007,-0.0052,0.0060,-0.0002,0.0073,0.0072
2026-05-27,-0.0011,-0.0027,0.0014,-0.0006,-0.0042,-0.0045


,Growth,Low Volatility,Momentum,Quality,Size,Value
Date,,,,,,
2007-01-02,-0.0002,0.0009,0.0005,-0.0001,-0.0016,0.0026
2007-01-03,-0.0011,-0.0063,-0.0039,-0.0029,-0.0039,-0.0042
2007-01-04,-0.0011,-0.0069,-0.0073,-0.0017,-0.0002,-0.0047
2007-01-05,0.0019,-0.0027,-0.0043,0.0005,-0.0028,-0.0021
2007-01-08,0.0004,-0.0006,0.0000,0.0008,-0.0015,-0.0000
...,...,...,...,...,...,...
2026-05-25,0.0057,0.0126,0.0079,-0.0012,0.0000,0.0055
2026-05-26,0.0007,0.0060,0.0073,-0.0002,0.0072,-0.0052
2026-05-27,-0.0011,0.0014,-0.0042,-0.0006,-0.0045,-0.0027


#### 1期間OLS


In [4]:
def run_factor_regression(
    ret_quality,
    y_col="Quality",
    x_cols=("Value", "Size", "Momentum", "Low Volatility"),
):
    X = sm.add_constant(ret_quality[list(x_cols)])
    y = ret_quality[y_col]
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 21})

    return model


model_full = run_factor_regression(ret_quality)
print(model_full.summary())

                            OLS Regression Results                            
Dep. Variable:                Quality   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.600
Method:                 Least Squares   F-statistic:                     132.7
Date:                Mon, 08 Jun 2026   Prob (F-statistic):          7.06e-108
Time:                        13:14:20   Log-Likelihood:                 24490.
No. Observations:                4995   AIC:                        -4.897e+04
Df Residuals:                    4990   BIC:                        -4.894e+04
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           4.907e-05    3.1e-05      1.

### Rolling collinearity


In [58]:
vif_ts, cond_ts = rolling_collinearity(
    ret_quality[[c for c in ret_quality.columns if c != "Quality"]],
)

# plot
fig = go.Figure()
for col in vif_ts.columns:
    fig.add_trace(go.Scatter(x=vif_ts.index, y=vif_ts[col], name=col, mode="lines"))

fig_title = (
    f"{target_sector}: Rolling collinearity (VIF)"
    if target_sector
    else "Rolling collinearity (VIF)"
)
fig.update_layout(
    template="plotly_dark",
    margin=dict(l=40, t=50, b=40, r=40),
    title=fig_title,
)
fig.update_xaxes(title="Date")
fig.show()

### Rolling correlation


In [61]:
window = 256
corr_ts = rolling_correlation(
    ret_quality[[c for c in ret_quality.columns if c != "Quality"]], window=window
)

display(corr_ts.head())

# --- 主要ペアの抽出 ---
peak_abs = corr_ts.abs().max().sort_values(ascending=False)  # 最大絶対相関
sign_flip = corr_ts.apply(lambda s: (s.min() < 0) and (s.max() > 0))  # 符号反転の有無
span = (corr_ts.max() - corr_ts.min()).sort_values(ascending=False)  # 変動レンジ

# print("=== 最大絶対相関 Top5 ===")
# display(peak_abs.head(5).round(2))
# print("\n=== 符号反転したペア ===")
# display(sign_flip[sign_flip].index.tolist())
# print("\n=== 変動レンジ Top5 ===")
# display(span.head(5).round(2))

# --- プロット: 変動の大きい主要ペアのみ ---
key_pairs = span.head(6).index.tolist()
fig = go.Figure()
for col in key_pairs:
    fig.add_trace(go.Scatter(x=corr_ts.index, y=corr_ts[col], name=col, mode="lines"))

fig_title = (
    f"{target_sector} Factor LS Return: Rolling correlation (window={window} days)"
    if target_sector
    else f"Factor LS Return: Rolling correlation (window={window} days)"
)
fig.update_layout(
    title=fig_title,
    template="plotly_dark",
    margin=dict(l=40, t=40, b=40, r=40),
)
fig.show()

,Growth × Low Volatility,Growth × Momentum,Growth × Size,Growth × Value,Low Volatility × Momentum,Low Volatility × Size,Low Volatility × Value,Momentum × Size,Momentum × Value,Size × Value
2008-01-03,0.016376,-0.051294,-0.078455,0.075868,-0.092653,-0.343300,0.277480,0.159502,-0.143063,-0.180710
2008-01-04,0.015723,-0.053551,-0.084422,0.084962,-0.084931,-0.341723,0.280067,0.159054,-0.139128,-0.172994
2008-01-07,0.012251,-0.063963,-0.103786,0.095395,-0.079864,-0.328344,0.277444,0.182462,-0.151776,-0.183861
2008-01-08,0.020433,-0.059167,-0.115597,0.092908,-0.078851,-0.336963,0.276322,0.179685,-0.150518,-0.182010
2008-01-09,0.036610,-0.062513,-0.136481,0.102952,-0.089706,-0.360065,0.281220,0.192388,-0.158594,-0.188567


### ファクターLSのボラティリティ


In [15]:
window = 256
vol_factors = (
    ret_quality.rolling(window=window, min_periods=window).std() * np.sqrt(252)
).dropna(how="all")

vol_factors.rename(
    columns={
        key: value
        for key, value in zip(
            vol_factors.columns, [c + "_vol" for c in vol_factors.columns], strict=False
        )
    },
    inplace=True,
)


if target_sector:
    df_cum_merge = df_cum[
        [
            f"FTW MXWD Index {target_sector} Quality Long-Short (High-Low) Total Return",
            f"FTW MXWD Index {target_sector} Value Long-Short (Low-High) Total Return",
        ]
    ]
else:
    df_cum_merge = df_cum[
        [
            "FTW MXWD Index Quality Long-Short (High-Low) Total Return",
            "FTW MXWD Index Value Long-Short (Low-High) Total Return",
        ]
    ]

vol_factors = pd.merge(
    vol_factors,
    df_cum_merge,
    left_index=True,
    right_index=True,
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
for col in [
    c
    for c in vol_factors.columns
    if (c.endswith("_vol")) & any(f in c for f in ["Quality", "Value"])
]:
    fig.add_trace(
        go.Scatter(x=vol_factors.index, y=vol_factors[col], name=col, mode="lines"),
        secondary_y=False,
    )

for col in [c for c in vol_factors.columns if not c.endswith("_vol")]:
    if "Quality" in col:
        color = "white"
    elif "Value" in col:
        color = "gray"
    fig.add_trace(
        go.Scatter(
            x=vol_factors.index,
            y=vol_factors[col],
            name=col,
            mode="lines",
            line=dict(color=color, width=2.5),
        ),
        secondary_y=True,
    )


fig.update_layout(
    title=f"Factor Return and Rolling Volatility (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)
fig.update_yaxes(title_text="Annualized Volatility", secondary_y=False)
fig.update_yaxes(title_text="Factor LS Total Return", secondary_y=True)


fig.show()

#### ローリングでOLS


In [65]:
window = 256
df_rolling_res = run_rolling_factor_regression(
    df_daily_return=ret_quality, window=window, maxlags=21
)


fig = make_subplots(specs=[[{"secondary_y": True}]])

# ファクターベータと決定係数の時系列推移
coef_cols = [c for c in df_rolling_res.columns if "coef_" in c and "const" not in c]
# coef_cols = ["coef_Value", "coef_Growth"]
for col in coef_cols:
    fig.add_trace(
        go.Scatter(
            x=df_rolling_res.index,
            y=df_rolling_res[col],
            mode="lines",
            name=col.replace("coef_", ""),
            opacity=0.7,
        ),
        secondary_y=False,
    )

fig.add_trace(
    go.Scatter(
        x=df_rolling_res.index,
        y=df_rolling_res["coef_const"] * 252,
        mode="lines",
        name="Annualized Alpha",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=df_rolling_res.index,
        y=df_rolling_res["R_squared"],
        mode="lines",
        name="R-squared(RHS)",
        line=dict(color="white", width=2.5, dash="dash"),
    ),
    secondary_y=True,
)


fig.update_layout(
    title=f"Rolling Factor Betas(LHS) and R-squared(RHS) (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)

fig.update_yaxes(title_text="Beta Coefficient", secondary_y=False)
fig.update_yaxes(title_text="R-squared", range=[0, 1], secondary_y=True)

fig.show()

## 複数のウィンドウサイズでのalphaをcheck


In [64]:
window = 256
df_rolling_res = run_rolling_factor_regression(
    df_daily_return=ret_quality, window=window, maxlags=21
)
df_rolling_res_with_return = pd.merge(
    df_rolling_res,
    df_cum[
        [
            # "FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return"
            "FTW MXWD Index Quality Long-Short (High-Low) Total Return"
        ]
    ],
    left_index=True,
    right_index=True,
)


fig = make_subplots(specs=[[{"secondary_y": True}]])

# ファクターベータと決定係数の時系列推移
# coef_cols = [c for c in df_rolling_res.columns if "coef_" in c and "const" not in c]
# coef_cols = ["coef_Value", "coef_Growth"]
# for col in coef_cols:
#     fig.add_trace(
#         go.Scatter(
#             x=df_rolling_res.index,
#             y=df_rolling_res[col],
#             mode="lines",
#             name=col.replace("coef_", ""),
#             opacity=0.7,
#         ),
#         secondary_y=False,
#     )

fig.add_trace(
    go.Scatter(
        x=df_rolling_res_with_return.index,
        y=df_rolling_res_with_return["coef_const"] * 252,
        mode="lines",
        name="Annualized Alpha",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=df_rolling_res_with_return.index,
        y=df_rolling_res_with_return[
            # "FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return"
            "FTW MXWD Index Quality Long-Short (High-Low) Total Return"
        ],
        mode="lines",
        name="Quality LS Total Return(RHS)",
        line=dict(color="white"),
    ),
    secondary_y=True,
)


fig.update_layout(
    title=f"Annualized Alpha(LHS) and Quality LS Total Return(RHS) (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)

fig.update_yaxes(title_text="Annualized Alpha", secondary_y=False)
fig.update_yaxes(title_text="Factor Return", secondary_y=True)

fig.show()